<div align="center">

# YAPAY ZEKÂ DESTEKLİ İLAÇ TASARIMI

**Erzincan Binali Yıldırım Üniversitesi — Sürekli Eğitim Uygulama ve Araştırma Merkezi**

28–29 Kasım 2026 · ML tabanlı QSAR uygulamalı eğitimi

</div>

---

### Eğitim sorumlusu

**Dr. Öğr. Üyesi Ekrem Yaşar**
Erzincan Binali Yıldırım Üniversitesi, Tıp Fakültesi, Biyofizik Anabilim Dalı

📧 `ey.biophysics@gmail.com`

---

# Oturum 2 — Molekülü Sayıya Çevirmek: Betimleyiciler ve Parmak İzleri

> **Girdi:** `ntrk1_regresyon.csv` (Oturum 1'in çıktısı)
> **Çıktı:** `ntrk1_descriptors.csv`, `ntrk1_ecfp4.csv`, `ntrk1_maccs.csv`

Oturum 1'in sonunda elimizde iki kolonlu bir tablo kaldı: `Smiles` ve
`pIC50`. Ama `Smiles` bir **metindir** —
`"Cn1ccc(-c2cc(C(=O)N...)"` gibi. Hiçbir makine öğrenmesi algoritması metinden
doğrudan öğrenemez; hepsi **sayı matrisi** ister.

Bu not defterinin tek işi o dönüşümü yapmaktır:

```
"CC(=O)Oc1ccccc1C(=O)O"   ──►   [180.16, 1.31, 1, 3, 63.6, ... ]
       (metin)                        (sayi vektoru)
```

## Molekülü sayıya çevirmenin iki yolu

Bu dönüşüme **moleküler temsil** (molecular representation) denir ve QSAR'ın
kalbidir. İki tamamlayıcı yaklaşım kullanacağız:

| | **Betimleyiciler** (descriptors) | **Parmak izleri** (fingerprints) |
|---|---|---|
| **Ne kodlar?** | Molekülün bütününe ait ölçülebilir özellikler | Molekül içindeki alt-yapıların varlığı |
| **Örnek** | `MolWt = 180.16`, `MolLogP = 1.31` | `FP_42 = 1` (şu alt-yapı var) |
| **Boyut** | ~200 sayı | 2048 bit (0/1) |
| **Yorumlanabilirlik** | **Yüksek** — her sayının kimyasal anlamı var | Düşük — her bit soyut bir alt-yapı |
| **Güçlü olduğu yer** | Fizikokimyasal eğilimler, XAI/SHAP | Yapısal örüntü ve benzerlik |
| **Ölçekleme** | Gerekir (çok farklı birimler) | Gerekmez (zaten 0/1) |

İkisini de üreteceğiz ve Oturum 3'te **hangisinin daha iyi tahmin ettiğini**
karşılaştıracağız. Bu, kursun deneysel sorularından biridir: *yorumlanabilir
fizikokimyasal özellikler mi, yoksa ham yapısal örüntüler mi?*

> **Üçüncü bir temsil** olarak **MACCS anahtarlarını** da üreteceğiz: 167 adet,
> elle tanımlanmış, **anlamı bilinen** alt-yapı sorusu ("molekülde kükürt var mı?",
> "aromatik halka var mı?"). Yorumlanabilirlik ile yapısal kodlama arasında bir
> orta yoldur.

---
## 1. Yapılandırma

`CORR_ESIGI`, birbiriyle neredeyse aynı bilgiyi taşıyan betimleyicileri elemek
için kullanılacak korelasyon eşiğidir. QSAR literatüründe yaygın aralık
**0.90–0.95**'tir; biz bilgi kaybını en aza indiren **0.95**'i seçiyoruz.

`FP_YARICAP = 2` ve `FP_BIT = 2048`, literatürde **ECFP4** olarak bilinen
standart Morgan parmak izi ayarıdır. (Adlandırma şaşırtıcıdır: yarıçap 2,
*çap* 4 demektir — ECFP**4** adı buradan gelir.)

In [ ]:
# ── Hedef (Oturum 1 ile ayni) ─────────────────────────────────────────────────
TARGET_ADI  = "NTRK1"
HEDEF_KOLON = "pIC50"

# ── Google Drive proje klasoru (Oturum 1 ile AYNI) ────────────────────────────
DRIVE_PROJE_KLASORU = "/content/drive/MyDrive/EBYUSEM2026_MLQSAR"
YEREL_KLASOR        = "EBYUSEM2026_MLQSAR"
DEPO_RAW            = "https://raw.githubusercontent.com/eygpcr/ebyusem2026-mlqsar/main"

# ── Girdi / cikti dosya adlari ───────────────────────────────────────────────
AD_GIRDI  = "ntrk1_siniflandirma.csv"      # Oturum 1 ciktisi: Smiles, pIC50, Aktiflik
AD_DESC   = "ntrk1_descriptors.csv"     # RDKit 2D betimleyiciler (OLCEKLENMEMIS)
AD_ECFP   = "ntrk1_ecfp4.csv"     # Morgan ECFP4 parmak izleri
AD_MACCS  = "ntrk1_maccs.csv"    # MACCS anahtarlari

# ── Parmak izi parametreleri ─────────────────────────────────────────────────
FP_YARICAP = 2      # radius=2  ->  ECFP4
FP_BIT     = 2048

# ── Betimleyici temizlik esigi ───────────────────────────────────────────────
CORR_ESIGI = 0.95   # |r| bu esigin uzerindeki ciftlerden biri elenir

print(f"Hedef           : {TARGET_ADI}")
print(f"Parmak izi      : ECFP{2 * FP_YARICAP}, {FP_BIT} bit")
print(f"Korelasyon esigi: |r| > {CORR_ESIGI}")

---
## 2. Google Drive'ı bağlama

Oturum 1 ile **aynı** proje klasörünü bağlıyoruz: bu not defteri girdisini
oradan okur, çıktılarını oraya yazar.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJE = Path(DRIVE_PROJE_KLASORU)
    ORTAM = "Colab + Drive"
except (ImportError, ModuleNotFoundError):
    PROJE = Path(YEREL_KLASOR)
    ORTAM = "Yerel"

PROJE.mkdir(parents=True, exist_ok=True)

YOL_GIRDI = PROJE / AD_GIRDI
YOL_DESC  = PROJE / AD_DESC
YOL_ECFP  = PROJE / AD_ECFP
YOL_MACCS = PROJE / AD_MACCS

print(f"Ortam         : {ORTAM}")
print(f"Proje klasoru : {PROJE}")

---
## 3. Kurulum ve içe aktarma

In [ ]:
!pip install rdkit -q
print("Kurulum tamam.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Draw, MACCSkeys, rdFingerprintGenerator
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

RDLogger.DisableLog("rdApp.*")
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110

print("Kutuphaneler yuklendi.")

---
## 4. Oturum 1'in veri setini yükleme

Veriyi önce Drive'dan okumayı deniyoruz. Bulunamazsa (ör. Oturum 1'i
çalıştırmadıysanız ya da yeni bir çalışma zamanı açtıysanız) kurs deposundaki
hazır dosyaya düşüyoruz. Böylece bu not defteri **tek başına** da çalışır.

In [ ]:
if YOL_GIRDI.exists():
    df = pd.read_csv(YOL_GIRDI)
    print(f"Drive'dan okundu: {YOL_GIRDI}")
else:
    url = f"{DEPO_RAW}/03_veri/{AD_GIRDI}"
    df = pd.read_csv(url)
    print(f"Drive'da bulunamadi; depodan okundu:\n  {url}")
    print("  (Oturum 1'i calistirdiysaniz ayni klasoru kullandiginizdan emin olun.)")

df = df.dropna(subset=["Smiles", HEDEF_KOLON]).reset_index(drop=True)

print(f"\nVeri seti sekli : {df.shape}")
print(f"Kolonlar        : {list(df.columns)}")
print(f"{HEDEF_KOLON} araligi  : {df[HEDEF_KOLON].min():.2f} - {df[HEDEF_KOLON].max():.2f}")
df.head()

---
## 5. SMILES → RDKit molekül nesnesi

Hem betimleyiciler hem parmak izleri, SMILES metninden değil bir RDKit **`Mol`**
nesnesinden hesaplanır. `Mol`, molekülün atomlarını, bağlarını ve
aromatikliğini içeren yapısal temsildir.

Oturum 1'de SMILES'ları kanonikleştirip geçersizleri elemiştik; yine de burada
savunmacı bir **geçerlilik kontrolü** yapıyoruz. `Chem.MolFromSmiles`
ayrıştıramadığı bir SMILES için `None` döndürür.

> **İyi alışkanlık:** Her adımda girdinin beklediğiniz biçimde olduğunu
> doğrulayın. Sessizce yanlış sonuç üretmek, hata vermekten daha kötüdür.

In [ ]:
df["mol"] = df["Smiles"].apply(Chem.MolFromSmiles)

once = len(df)
df = df[df["mol"].notna()].reset_index(drop=True)

print(f"Gecerli molekul  : {len(df)}")
print(f"Elenen (gecersiz): {once - len(df)}")

# Atom sayisi dagilimina hizli bir bakis
atom_sayilari = df["mol"].apply(lambda m: m.GetNumAtoms())
print(f"\nAgir atom sayisi : ortalama {atom_sayilari.mean():.1f}, "
      f"aralik {atom_sayilari.min()}-{atom_sayilari.max()}")

---
## 6. Ara durak: Lipinski kuralları ve ilaç benzerliği

Bütün betimleyicileri hesaplamadan önce, bunların **en tanınmış beşini**
ayrıca ele alalım. Christopher Lipinski'nin 1997'de önerdiği **"Beşler Kuralı"**
(Rule of Five), ağızdan alınan ilaçların paylaştığı fizikokimyasal eğilimleri
özetler:

| Özellik | Kural | Anlamı |
|---|---|---|
| `MolWt` | ≤ 500 Da | Çok büyük moleküller bağırsaktan emilmez |
| `MolLogP` | ≤ 5 | Çok yağda çözünen moleküller toksisite ve çözünürlük sorunu çıkarır |
| `NumHDonors` | ≤ 5 | Çok fazla H-bağı verici zarı geçmeyi zorlaştırır |
| `NumHAcceptors` | ≤ 10 | Aynı gerekçe |

**Önemli uyarı:** Bu bir *kural* değil, bir *eğilimdir*. Kuralı ihlal eden pek
çok başarılı ilaç vardır (özellikle kinaz inhibitörleri ve doğal ürünler
sıklıkla 500 Da'yı aşar). Lipinski'nin kendisi de bunu bir filtre değil, bir
uyarı işareti olarak sunmuştur.

Aşağıda hem bu beş özelliği hesaplayıp hem de veri setimizin kaç ihlal
taşıdığına bakıyoruz. **Kinaz inhibitörleri için ihlallerin yaygın olmasını
bekliyoruz** — bu bir hata değil, hedef sınıfının doğasıdır.

In [ ]:
def lipinski_ozellikleri(mol):
    return pd.Series({
        "MolWt":         Descriptors.MolWt(mol),
        "MolLogP":       Descriptors.MolLogP(mol),
        "NumHDonors":    Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "TPSA":          Descriptors.TPSA(mol),
    })


df_lipinski = df["mol"].apply(lipinski_ozellikleri)

# Ihlal sayisi (0-4)
ihlal = (
    (df_lipinski["MolWt"] > 500).astype(int)
    + (df_lipinski["MolLogP"] > 5).astype(int)
    + (df_lipinski["NumHDonors"] > 5).astype(int)
    + (df_lipinski["NumHAcceptors"] > 10).astype(int)
)

print("Lipinski ozellikleri — ozet")
print("-" * 58)
print(df_lipinski.describe().loc[["mean", "std", "min", "max"]].round(2).to_string())

print("\nBesler Kurali ihlal dagilimi")
print("-" * 58)
for k, n in ihlal.value_counts().sort_index().items():
    print(f"  {k} ihlal : {n:>5} molekul  (%{100 * n / len(df):.1f})")
print(f"\nHic ihlal etmeyen (klasik 'ilac benzeri'): "
      f"%{100 * (ihlal == 0).mean():.1f}")

In [ ]:
# Lipinski ozellikleri aktiflikle iliskili mi?
fig, eksenler = plt.subplots(1, 4, figsize=(15, 3.6))
for eksen, ozellik in zip(eksenler, ["MolWt", "MolLogP", "NumHDonors", "TPSA"]):
    eksen.scatter(df_lipinski[ozellik], df[HEDEF_KOLON], s=5, alpha=0.25,
                  color="#4878a8")
    r = np.corrcoef(df_lipinski[ozellik], df[HEDEF_KOLON])[0, 1]
    eksen.set_xlabel(ozellik)
    eksen.set_title(f"r = {r:.3f}", fontsize=10)
    eksen.grid(alpha=0.3, linestyle="--")
eksenler[0].set_ylabel(HEDEF_KOLON)
plt.suptitle("Tek tek Lipinski ozellikleri aktifligi aciklamaya yetmez",
             y=1.04, fontsize=11)
plt.tight_layout()
plt.show()

print("Korelasyonlarin dusuk olmasi beklenen ve ogretici bir sonuctur:")
print("aktiflik tek bir fizikokimyasal ozellige indirgenemez. Bu yuzden")
print("yuzlerce betimleyiciyi birlikte kullanan bir MODELE ihtiyac duyuyoruz.")

---
## 7. RDKit 2D betimleyicilerinin tamamını hesaplama

RDKit, kutudan çıktığı hâliyle **200'den fazla** 2D moleküler betimleyici sunar.
Bunlar beş aileye ayrılabilir:

| Aile | Örnek | Ne ölçer? |
|---|---|---|
| **Kurucu (constitutional)** | `MolWt`, `HeavyAtomCount`, `RingCount` | Basit sayımlar |
| **Topolojik** | `Chi0v`, `Kappa1`, `BalabanJ` | Moleküler grafın bağlanma biçimi |
| **Fizikokimyasal** | `MolLogP`, `TPSA`, `MolMR` | Çözünürlük, polarite, kırılma |
| **Elektronik** | `MaxPartialCharge`, `PEOE_VSA*` | Yük dağılımı |
| **Fragman sayımları** | `fr_benzene`, `fr_amide` | Belirli fonksiyonel grup sayısı |

`Descriptors._descList` bütün betimleyici adlarını verir;
`MolecularDescriptorCalculator` bunların hepsini her molekül için tek seferde
hesaplar.

> ⏱️ Bu hücre birkaç bin molekül için **yarım ila iki dakika** sürebilir.

**Savunmacı hesaplama.** Bazı betimleyiciler belirli moleküllerde hata verebilir
ya da taşma (overflow) üretebilir — en bilinen örnek `Ipc` betimleyicisidir:
molekül büyüdükçe değeri astronomik boyutlara çıkar ve `inf` olur. Bu yüzden
hesaplamayı `try/except` içine alıyoruz ve sorunlu molekül için `NaN`
döndürüyoruz. Bu değerleri bir sonraki bölümde temizleyeceğiz.

In [ ]:
betimleyici_adlari = [ad for ad, _ in Descriptors._descList]
hesaplayici = MolecularDescriptorCalculator(betimleyici_adlari)

print(f"Hesaplanacak betimleyici sayisi: {len(betimleyici_adlari)}")


def betimleyicileri_hesapla(mol):
    """Bir molekul icin tum 2D betimleyicileri dondurur; hata olursa NaN dizisi."""
    try:
        return hesaplayici.CalcDescriptors(mol)
    except Exception:
        return (np.nan,) * len(betimleyici_adlari)


print("Hesaplaniyor... (birkac dakika surebilir)")
degerler = [betimleyicileri_hesapla(m) for m in df["mol"]]
X_desc = pd.DataFrame(degerler, columns=betimleyici_adlari, index=df.index)

print(f"Ham betimleyici matrisi: {X_desc.shape}")
X_desc.iloc[:5, :8]

---
## 8. Temizlik — Adım 1: her şeyi sayıya çevirmek

Bir modelin bütün girdileri **sayısal** olmalıdır. İki tür sorunu gideriyoruz:

1. **Mantıksal (boolean) kolonlar → tam sayı.** Bazı betimleyiciler
   `True`/`False` döndürebilir. Model için `True` doğrudan anlamlı değildir;
   `1`/`0` ise işlenebilir.
2. **Metinsel / karışık tipli kolonlar.** `pd.to_numeric(errors="coerce")` ile
   bütün kolonları sayıya zorluyoruz; çevrilemeyen hücreler `NaN` olur ve bir
   sonraki adımda temizlenir.

Bu iki işlem **deterministiktir** — veriye bakıp bir parametre öğrenmezler.
Bu yüzden bütün veri üzerinde güvenle uygulanabilirler ve **veri sızıntısı
riski taşımazlar**. (Bu ayrım Adım 4'te kritik hâle gelecek.)

In [ ]:
# 1) Mantiksal kolonlari 0/1'e cevir
bool_kolonlar = X_desc.select_dtypes(include=["bool"]).columns.tolist()
if bool_kolonlar:
    X_desc[bool_kolonlar] = X_desc[bool_kolonlar].astype(int)

# 2) Butun kolonlari sayisala zorla
X_desc = X_desc.apply(pd.to_numeric, errors="coerce")

print(f"Mantiksal -> 0/1 cevrilen kolon : {len(bool_kolonlar)}")
print(f"Matris sekli                    : {X_desc.shape}")
print(f"Sayiya cevrilemeyen hucre       : {int(X_desc.isna().sum().sum())}")

---
## 9. Temizlik — Adım 2: geçersiz ve sabit kolonları atmak

Üç tür işe yaramaz kolonu kaldırıyoruz:

**1. Geçersiz değer içerenler.** `±inf` değerlerini önce `NaN`'a çeviriyor,
ardından içinde `NaN` bulunan **kolonları** atıyoruz. Neden satır değil kolon?
Çünkü tek bir sorunlu betimleyici yüzünden bir molekülü (ve onun deneysel
aktivite ölçümünü) kaybetmek israftır; sorunlu betimleyiciyi atmak daha
akıllıdır.

**2. Sayısal olarak kullanılamayacak kadar büyük değerler.** Bu adım ilk
bakışta gereksiz görünür ama **kritiktir** — ve bu kursun hazırlığı sırasında
gerçek veride somut bir çökmeye yol açtığı için eklenmiştir.

Sorun şudur: bir değer `inf` olmadığı hâlde işe yaramayacak kadar büyük
olabilir. En bilinen örnek **`Ipc`** betimleyicisidir (bilgi içeriği indeksi):
molekül büyüdükçe değeri çarpıcı biçimde artar. Veri setimizde `Ipc` değerleri
10⁴ ile **10⁵⁷** arasında değişiyor.

Neden çöker? scikit-learn'ün ağaç tabanlı modelleri veriyi içeride
**`float32`**'ye çevirir ve `float32`'nin üst sınırı yaklaşık **3,4 × 10³⁸**'dir.
Bu sınırın üzerindeki değerler dönüşüm sırasında `inf` olur ve model
`ValueError: Input X contains infinity` hatasıyla durur. Üstelik bu hata
Oturum 2'de değil, **Oturum 3'te model eğitilirken** ortaya çıkar — yani
sorunun kaynağından çok uzakta.

Bu yüzden mutlak değeri `10³⁰`'u aşan kolonları atıyoruz. Bu eşik `float32`
sınırının çok altındadır; böylece ölçekleme ve korelasyon hesaplarında da
güvenli bir pay bırakılır. Zaten bu büyüklükte bir betimleyici kimyasal olarak
yorumlanabilir bir bilgi taşımaz.

> **Alınacak ders.** Veri temizliğinde `NaN` ve `inf` kontrolü yetmez;
> **değer aralığının** modelin sayısal gösterimine sığıp sığmadığı da
> denetlenmelidir.

**3. Sabit (sıfır varyanslı) kolonlar.** Bütün moleküllerde aynı değeri taşıyan
bir betimleyici hiçbir ayırt edici bilgi vermez. Örneğin veri setimizdeki
hiçbir molekülde berilyum yoksa `fr_Be` kolonu baştan sona sıfırdır.

Her adımda kaç kolonun neden atıldığını yazdırıyoruz — hangi betimleyicilerin
elendiğini bilmek, sonradan sonuçları yorumlarken işinize yarar.

In [ ]:
kolon_once = X_desc.shape[1]

# ── 1) inf -> NaN, sonra NaN iceren kolonlari at ─────────────────────────────
X_desc = X_desc.replace([np.inf, -np.inf], np.nan)
nan_kolonlar = X_desc.columns[X_desc.isna().any()].tolist()
X_desc = X_desc.drop(columns=nan_kolonlar)

# ── 2) float32'ye sigmayacak kadar buyuk kolonlari at ───────────────────────
# Sinir: float32 ust siniri ~3.4e38. Guvenli bir pay birakmak icin 1e30.
BUYUK_SINIR = 1e30
en_buyuk = X_desc.abs().max()
buyuk_kolonlar = en_buyuk[en_buyuk > BUYUK_SINIR].index.tolist()
X_desc = X_desc.drop(columns=buyuk_kolonlar)

# ── 3) Sabit (sifir varyansli) kolonlari at ────────────────────────────────
varyanslar = X_desc.var(numeric_only=True)
sabit_kolonlar = varyanslar[varyanslar == 0].index.tolist()
X_desc = X_desc.drop(columns=sabit_kolonlar)

print(f"Baslangic betimleyici sayisi      : {kolon_once}")
print(f"1) Gecersiz (NaN/inf) atilan      : {len(nan_kolonlar)}")
if nan_kolonlar:
    print(f"     ornekler: {nan_kolonlar[:6]}")
print(f"2) Asiri buyuk (>{BUYUK_SINIR:.0e}) atilan : {len(buyuk_kolonlar)}")
for ad in buyuk_kolonlar:
    print(f"     {ad}: en buyuk mutlak deger {en_buyuk[ad]:.3e}")
print(f"3) Sabit (sifir varyansli) atilan : {len(sabit_kolonlar)}")
if sabit_kolonlar:
    print(f"     ornekler: {sabit_kolonlar[:6]}")
print(f"Kalan betimleyici                 : {X_desc.shape[1]}")

# Guvence: kalan butun degerler float32'ye donusturulebiliyor mu?
guvenli = np.isfinite(X_desc.values.astype(np.float32)).all()
assert guvenli, "Hala float32'ye sigmayan deger var!"
print("\nfloat32 uyumluluk kontrolu: GECTI")

---
## 10. Temizlik — Adım 3: yüksek korelasyonlu betimleyicileri elemek

Moleküler betimleyicilerin birçoğu **aynı şeyi iki kez** ölçer. Örneğin
moleküler ağırlık ile ağır atom sayısı neredeyse birebir ilişkilidir — biri
büyüdüğünde diğeri de büyür.

Bu duruma **çoklu-doğrusallık** (multicollinearity) denir ve üç sorun yaratır:

1. **Kararsızlık.** Lineer modellerde katsayılar güvenilmez hâle gelir: aynı
   bilgiyi taşıyan iki betimleyici arasında katsayı rastgele paylaşılır.
2. **Yorumlanabilirlik kaybı.** SHAP analizinde (Oturum 5) önem, birbirinin
   kopyası olan betimleyiciler arasında bölünür ve hiçbiri önemli görünmez.
3. **Gereksiz boyut.** Daha fazla kolon, daha fazla hesap ve daha fazla aşırı
   öğrenme riski demektir.

**Yöntem:** Betimleyiciler arası mutlak Pearson korelasyon matrisinin yalnızca
**üst üçgenine** bakılır. Bir kolon, kendinden önceki herhangi bir kolonla
`|r| > 0.95` ilişki taşıyorsa atılır. Böylece her yüksek-korelasyonlu gruptan
**tek bir temsilci** kalır.

> **Not.** Pearson korelasyonu ölçekten bağımsızdır, bu yüzden bu adımı
> ölçeklemeden önce yapmak sonucu değiştirmez.

In [ ]:
korelasyon = X_desc.corr().abs()

# Yalnizca ust ucgen (kosegen ve alt ucgen haric)
ust_ucgen = korelasyon.where(
    np.triu(np.ones(korelasyon.shape), k=1).astype(bool)
)

atilacak = [kolon for kolon in ust_ucgen.columns
            if (ust_ucgen[kolon] > CORR_ESIGI).any()]

kolon_once_corr = X_desc.shape[1]
X_desc = X_desc.drop(columns=atilacak)

print(f"Korelasyon esigi (|r| >)         : {CORR_ESIGI}")
print(f"Elenen yuksek-korelasyonlu kolon : {len(atilacak)}")
print(f"Filtre oncesi                    : {kolon_once_corr}")
print(f"Filtre sonrasi (NIHAI)           : {X_desc.shape[1]}")
print(f"\nElenenlere ornek: {atilacak[:8]}")

---
## 11. Betimleyici veri setini kaydetme — ve ölçekleme hakkında kritik bir not

Betimleyici matrisimiz hazır. Şimdi onu `Smiles` ve hedef kolonlarla birleştirip
kaydediyoruz.

### Niçin **ölçeklenmemiş** kaydediyoruz?

`MolWt` yüzler mertebesinde, `MolLogP` birler mertebesindedir. SVR, kNN ve
lineer modeller gibi **mesafe/ölçek duyarlı** algoritmalar bu farktan rahatsız
olur ve ölçekleme ister.

Buna rağmen burada **ölçeklenmiş bir dosya üretmiyoruz.** Gerekçesi önemlidir:

> **`StandardScaler`'ı bütün veriye uydurmak veri sızıntısıdır.**
> Ölçekleyici, her kolonun **ortalamasını ve standart sapmasını** veriden
> *öğrenir*. Bu istatistikleri eğitim **ve** test verisinin tamamından
> hesaplarsanız, test verisinin dağılımına ait bilgi eğitim sürecine sızar.
> Sonuç: test skorunuz gerçekte olduğundan **iyi** görünür.

Doğru yol, ölçeklemeyi bir **`Pipeline`** içine koymaktır. O zaman
çapraz doğrulamanın her katmanında ölçekleyici yalnızca o katmanın eğitim
kısmına uydurulur. Oturum 3 ve 4'te tam olarak bunu yapacağız:

```python
Pipeline([("olcekle", StandardScaler()), ("model", SVR())])
```

Bu yüzden diske **ham (ölçeklenmemiş)** değerleri yazıyoruz: ölçekleme, modelin
bir parçasıdır — verinin değil.

> **Adım 1–3 neden sızıntı değil?** Mantıksal dönüşüm ve sayıya çevirme veriden
> hiçbir şey öğrenmez. Sabit kolon ve korelasyon filtreleri ise veriye bakar;
> teknik olarak bunları da eğitim katmanı içine almak en titiz yoldur. Ancak
> bunlar **hedef değişkeni (`pIC50`) hiç kullanmadığından** etkileri
> ihmal edilebilir kabul edilir ve QSAR pratiğinde veri setinin tamamında
> uygulanır. Ölçekleme için ise durum farklıdır çünkü etkisi doğrudan model
> tahminine girer.

In [ ]:
df_desc = pd.concat(
    [df[["Smiles"]].reset_index(drop=True),
     X_desc.reset_index(drop=True),
     df[[HEDEF_KOLON, "Aktiflik"]].reset_index(drop=True)],
    axis=1,
)

df_desc.to_csv(YOL_DESC, index=False)

print(f"Kaydedildi : {YOL_DESC}")
print(f"Sekil      : {df_desc.shape}")
print(f"             -> {X_desc.shape[1]} betimleyici + Smiles + "
      f"{HEDEF_KOLON} + Aktiflik")
print(f"Boyut      : {YOL_DESC.stat().st_size / 1024 / 1024:.1f} MB")
df_desc.iloc[:5, :6]

---
## 12. Morgan parmak izleri (ECFP4)

Şimdi ikinci temsile geçiyoruz. **Morgan parmak izi**, molekülün içindeki
**alt-yapıların varlığını** sabit uzunlukta bir `0/1` vektörüne kodlar.

**Nasıl çalışır?**

1. Her atom için, çevresindeki komşuluk `FP_YARICAP` katman derinliğine kadar
   taranır. Yarıçap 0'da yalnızca atomun kendisi, yarıçap 1'de komşuları,
   yarıçap 2'de komşularının komşuları...
2. Her alt-yapı bir sayıya (hash) dönüştürülür.
3. Bu sayı, `FP_BIT` uzunluğundaki vektörde bir konuma karşılık gelir ve o bit
   `1` yapılır.

Sonuç: "bu molekülde şu şu alt-yapılar var" diyen 2048 uzunluğunda bir bit
vektörü. Literatürde bu yaklaşım **ECFP** (Extended Connectivity FingerPrint)
olarak bilinir ve QSAR'da en çok kullanılan temsildir.

**Hash çakışması (bit collision).** 2048 bite sığdırılan alt-yapı sayısı çok
daha fazla olabilir; bu yüzden farklı alt-yapılar aynı bite düşebilir. Bit
sayısını artırmak (4096, 8192) çakışmayı azaltır ama boyutu büyütür. 2048 yaygın
bir dengedir.

> **Güncel API.** `rdFingerprintGenerator.GetMorganGenerator` kullanıyoruz.
> Eskiden yaygın olan `AllChem.GetMorganFingerprintAsBitVect` artık önerilmiyor;
> yeni üreteç arayüzü hem daha hızlı hem de parmak izi türleri arasında tutarlı.

In [ ]:
morgan = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_YARICAP, fpSize=FP_BIT
)


def ecfp_hesapla(mol):
    """Molekulun Morgan bit vektorunu NumPy dizisi olarak dondurur."""
    bit_vektoru = morgan.GetFingerprint(mol)
    dizi = np.zeros((FP_BIT,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(bit_vektoru, dizi)
    return dizi


print(f"ECFP{2 * FP_YARICAP} hesaplaniyor...")
ecfp_matrisi = np.vstack([ecfp_hesapla(m) for m in df["mol"]])
X_ecfp = pd.DataFrame(ecfp_matrisi,
                      columns=[f"FP_{i}" for i in range(FP_BIT)],
                      index=df.index)

print(f"Parmak izi matrisi: {X_ecfp.shape}")
X_ecfp.iloc[:5, :14]

### Parmak izi matrisi ne kadar seyrek?

Parmak izleri **seyrek** (sparse) matrislerdir: bitlerin büyük bölümü sıfırdır.
Aşağıdaki sayılar bunu gösterir. Üç gözlem önemlidir:

- **Molekül başına kaç bit `1`?** Tipik olarak 40–80 arası. Yani 2048 bitin
  yalnızca küçük bir kısmı kullanılır.
- **Hiç kullanılmayan bitler.** Veri setimizdeki hiçbir molekülde görülmeyen
  alt-yapılara karşılık gelen bitler baştan sona sıfırdır — betimleyicilerdeki
  "sabit kolon" durumunun aynısı.
- **Yalnızca bir molekülde görülen bitler.** Bunlar aşırı öğrenmeye açıktır:
  model, tek bir moleküle özgü bir bitten kural çıkarabilir.

**Parmak izlerini temizliyor muyuz?** Hayır — oldukları gibi bırakıyoruz.
Nedeni: ağaç tabanlı modeller sıfır varyanslı kolonları zaten görmezden gelir ve
hangi bitlerin "kullanışsız" olduğu eğitim bölünmesine göre değişir. Bitleri
bütün veriye bakarak atmak da bir tür sızıntı olurdu.

In [ ]:
bit_toplamlari = X_ecfp.sum(axis=0)
molekul_basina = X_ecfp.sum(axis=1)

print("ECFP4 matrisinin seyreklik profili")
print("-" * 54)
print(f"  Matris boyutu            : {X_ecfp.shape[0]} x {X_ecfp.shape[1]}")
print(f"  Doluluk orani            : %{100 * X_ecfp.values.mean():.2f}")
print(f"  Molekul basina dolu bit  : ortalama {molekul_basina.mean():.1f}"
      f"  (aralik {molekul_basina.min()}-{molekul_basina.max()})")
print(f"  Hic kullanilmayan bit    : {int((bit_toplamlari == 0).sum())} / {FP_BIT}")
print(f"  Tek molekulde gorulen bit: {int((bit_toplamlari == 1).sum())}")
print(f"  En sik gorulen bit       : {int(bit_toplamlari.max())} molekulde")

---
## 13. MACCS anahtarları (167 bit)

Üçüncü temsil: **MACCS anahtarları**. Morgan parmak izinden farkı şudur —
Morgan bitleri algoritmanın ürettiği *soyut* alt-yapılardır ve anlamları
baştan bilinmez. MACCS bitleri ise **elle tanımlanmış, anlamı bilinen 167
yapısal sorudan** oluşur:

- "Molekülde kükürt var mı?"
- "Aromatik halka var mı?"
- "Bir karbonile komşu azot var mı?"

Bu, onları **yorumlanabilir** kılar: bir model "MACCS bit 118 önemli" derse,
o bitin hangi yapısal soruya karşılık geldiğini tabloya bakıp okuyabilirsiniz.

Dezavantajı: 167 soru sınırlı bir kimyasal uzayı kapsar ve ince yapısal
farkları Morgan kadar iyi ayırt edemez. Bu yüzden üçünü birlikte üretip Oturum
3'te **karşılaştıracağız**.

In [ ]:
def maccs_hesapla(mol):
    bit_vektoru = MACCSkeys.GenMACCSKeys(mol)
    dizi = np.zeros((167,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(bit_vektoru, dizi)
    return dizi


print("MACCS anahtarlari hesaplaniyor...")
maccs_matrisi = np.vstack([maccs_hesapla(m) for m in df["mol"]])
X_maccs = pd.DataFrame(maccs_matrisi,
                       columns=[f"MACCS_{i}" for i in range(167)],
                       index=df.index)

print(f"MACCS matrisi    : {X_maccs.shape}")
print(f"Doluluk orani    : %{100 * X_maccs.values.mean():.1f}")
print(f"Kullanilmayan bit: {int((X_maccs.sum(axis=0) == 0).sum())} / 167")
X_maccs.iloc[:5, :14]

---
## 14. Parmak izi veri setlerini kaydetme

İki parmak izi matrisini de, betimleyicilerle **aynı düzende** kaydediyoruz:
`Smiles` solda, özellikler ortada, hedefler (`pIC50`, `Aktiflik`) sağda.
Bu tutarlılık Oturum 3 ve 4'te üç özellik setini birbirinin yerine kullanmayı
kolaylaştırır.

In [ ]:
def kaydet(X, yol, ad):
    tablo = pd.concat(
        [df[["Smiles"]].reset_index(drop=True),
         X.reset_index(drop=True),
         df[[HEDEF_KOLON, "Aktiflik"]].reset_index(drop=True)],
        axis=1,
    )
    tablo.to_csv(yol, index=False)
    mb = yol.stat().st_size / 1024 / 1024
    print(f"  {ad:<26} {str(tablo.shape):<16} {mb:>6.1f} MB   {yol.name}")
    return tablo


print("Kaydedilen ozellik setleri")
print("-" * 76)
df_ecfp  = kaydet(X_ecfp,  YOL_ECFP,  f"ECFP{2 * FP_YARICAP} ({FP_BIT} bit)")
df_maccs = kaydet(X_maccs, YOL_MACCS, "MACCS (167 bit)")
print(f"\nKlasor: {PROJE}")

---
## 15. Kimyasal uzayı görselleştirmek

Elimizde 4000'den fazla molekül ve her biri için yüzlerce sayı var. Bu çok
boyutlu bulutu **iki boyuta indirip** gözle bakabiliriz. Bunun için
**temel bileşen analizi** (PCA) kullanıyoruz: verinin en çok değiştiği iki yönü
bulur ve molekülleri o düzleme izdüşürür.

Noktaları `pIC50` değerine göre renklendiriyoruz. Bakarken şunu sorun:

> **Potent moleküller kimyasal uzayda belirli bölgelerde mi kümeleniyor?**

- **Eğer kümeleniyorsa:** yapı ile aktivite arasında öğrenilebilir bir ilişki
  var — QSAR modeli için iyi haber.
- **Eğer renkler tamamen karışıksa:** ya temsil yetersizdir, ya aktivite bu
  betimleyicilerle açıklanamıyordur (ör. farklı bağlanma modları), ya da veri
  çok heterojendir.

> **Dikkat.** Buradaki `StandardScaler` yalnızca **görselleştirme** içindir;
> hiçbir model eğitmiyoruz, dolayısıyla sızıntı sorunu yoktur. Model eğitiminde
> ölçekleme `Pipeline` içinde olacaktır (Oturum 3).

In [ ]:
# Gorsellestirme icin olcekleme (model EGITILMIYOR -> sizinti yok)
X_olcekli = StandardScaler().fit_transform(X_desc.values)
pca = PCA(n_components=2, random_state=0)
izdusum = pca.fit_transform(X_olcekli)

fig, (sol, sag) = plt.subplots(1, 2, figsize=(13.5, 5.2))

# --- Sol: surekli pIC50 ---
nokta = sol.scatter(izdusum[:, 0], izdusum[:, 1], c=df[HEDEF_KOLON],
                    cmap="viridis", s=9, alpha=0.65)
plt.colorbar(nokta, ax=sol, label=HEDEF_KOLON)
sol.set_xlabel(f"PC1  (varyansin %{100 * pca.explained_variance_ratio_[0]:.1f}'i)")
sol.set_ylabel(f"PC2  (varyansin %{100 * pca.explained_variance_ratio_[1]:.1f}'i)")
sol.set_title("Kimyasal uzay — betimleyici PCA")
sol.grid(alpha=0.25, linestyle="--")

# --- Sag: aktif / inaktif ---
for deger, renk, etiket in [(0, "#95a5a6", "Inaktif"), (1, "#27ae60", "Aktif")]:
    maske = df["Aktiflik"] == deger
    sag.scatter(izdusum[maske, 0], izdusum[maske, 1], c=renk, s=9,
                alpha=0.55, label=f"{etiket} (n={int(maske.sum())})")
sag.set_xlabel("PC1")
sag.set_ylabel("PC2")
sag.set_title("Ayni uzay — sinif etiketlerine gore")
sag.legend()
sag.grid(alpha=0.25, linestyle="--")

plt.tight_layout()
plt.savefig(PROJE / "ntrk1_kimyasal_uzay_pca.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Ilk iki bilesen toplam varyansin "
      f"%{100 * pca.explained_variance_ratio_[:2].sum():.1f}'ini aciklar.")
print("Bu oranin dusuk olmasi normaldir: yuzlerce betimleyicinin bilgisi")
print("iki boyuta sigmaz. Grafik bir harita, kanit degildir.")

---
## 16. Üretilen özellik setlerinin özeti

Üç temsil de **aynı moleküller** ve **aynı hedefler** üzerine kuruludur;
yalnızca molekülü sayıya çevirme biçimleri farklıdır. Oturum 3 ve 4'te bunları
yan yana karşılaştıracağız.

In [ ]:
ozet = pd.DataFrame([
    {"Ozellik seti": "RDKit betimleyiciler",
     "Ozellik sayisi": X_desc.shape[1],
     "Tur": "surekli sayi",
     "Olcekleme": "Pipeline icinde gerekir",
     "Yorumlanabilirlik": "yuksek",
     "Dosya": YOL_DESC.name},
    {"Ozellik seti": f"Morgan ECFP{2 * FP_YARICAP}",
     "Ozellik sayisi": FP_BIT,
     "Tur": "ikili (0/1)",
     "Olcekleme": "gerekmez",
     "Yorumlanabilirlik": "dusuk",
     "Dosya": YOL_ECFP.name},
    {"Ozellik seti": "MACCS anahtarlari",
     "Ozellik sayisi": 167,
     "Tur": "ikili (0/1)",
     "Olcekleme": "gerekmez",
     "Yorumlanabilirlik": "orta (bit anlamlari bilinir)",
     "Dosya": YOL_MACCS.name},
])

print(ozet.to_string(index=False))
print(f"\nHepsinde molekul sayisi: {len(df)}")
print(f"Hedefler: {HEDEF_KOLON} (regresyon) ve Aktiflik (siniflandirma)")

---
## 17. Özet

### Ne yaptık?

1. Molekülleri RDKit `Mol` nesnelerine çevirdik ve doğruladık.
2. **Lipinski** özelliklerine bakıp tek bir fizikokimyasal özelliğin aktifliği
   açıklamaya yetmediğini gördük — bu yüzden modele ihtiyacımız var.
3. **200'den fazla 2D betimleyici** hesapladık ve üç adımda temizledik:
   sayıya çevirme → geçersiz/sabit kolonları atma → yüksek korelasyon elemesi.
4. **Morgan ECFP4** (2048 bit) ve **MACCS** (167 bit) parmak izlerini ürettik.
5. Üç özellik setini de Drive'a kaydettik.
6. Kimyasal uzayı PCA ile görselleştirdik.

### Akılda kalması gerekenler

- **Temsil seçimi bir modelleme kararıdır.** Betimleyiciler yorumlanabilir,
  parmak izleri örüntü yakalamada güçlüdür. Hangisinin daha iyi olduğu veriye
  bağlıdır — bu yüzden ikisini de deneyeceğiz.
- **Ölçekleme verinin değil, modelin parçasıdır.** `StandardScaler`'ı bütün
  veriye uydurup kaydetmek sızıntıdır; doğru yer `Pipeline` içidir.
- **Çoklu-doğrusallık hem modeli hem yorumu bozar.** Korelasyon filtresi
  SHAP analizini (Oturum 5) anlamlı kılmak için de gereklidir.
- **Her temizlik adımında kaç kolonun neden atıldığını kaydedin.**

### Sırada ne var?

**Oturum 3 — Regresyon modelleri.** Artık sayı matrisimiz var. Sırada:
eğitim/test bölünmesi (rastgele **ve** iskelet bazlı), **LazyPredict** ile
onlarca modeli hızlıca tarama, **Optuna** ile hiperparametre optimizasyonu,
5 katlı çapraz doğrulama ve **y-randomizasyon** ile modelin gerçekten öğrenip
öğrenmediğini sınama.

---

### Alıştırmalar

1. `CORR_ESIGI`'ni 0.85'e çekin. Kaç betimleyici kalıyor? Oturum 3'te bu daha
   küçük özellik setinin model performansını düşürüp düşürmediğine bakın.
2. `FP_YARICAP`'ı 3 (ECFP6) ve `FP_BIT`'i 4096 yapın. Seyreklik profili ve
   "hiç kullanılmayan bit" sayısı nasıl değişiyor?
3. PCA yerine **t-SNE** veya **UMAP** ile kimyasal uzayı çizin. Kümelenme daha
   belirgin mi? (İpucu: `from sklearn.manifold import TSNE`)
4. En potent 20 molekülün PCA haritasındaki konumlarını işaretleyin. Belirli bir
   bölgede mi toplanıyorlar? Bu, Oturum 5'teki **uygulanabilirlik alanı**
   (applicability domain) tartışmasının habercisidir.